# Intro

## Global parameters

In [55]:
MAX_TIME_HOURS=1
ALPHA=0.01

In [ ]:
if platform.system() == 'Darwin':
    print('Air!')
    HOME = '/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/bowtie2/'
elif platform.system() == 'Linux':
    print('Stella!')
    HOME = '/home/sarawalk/bowtie2_py39/bowtie2/'
else:
    raise RuntimeError(f"Unsupported OS: {platform.system()}")


Stella!


In [ ]:
sys.path.insert(0, HOME)

## Modules

In [2]:
import os, pickle, platform, sys
import numpy as np

In [3]:
from collections import defaultdict

In [4]:
import dcms
from dcms.models import DCMModel, DECMModel, qDECMModel, DWCMModel

In [5]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [6]:
from scipy.stats import spearmanr

In [7]:
from tqdm.notebook import tqdm, trange

In [8]:
import datetime as dt

In [13]:
from auxiliary_functions import el2ks

In [14]:
from bowtie import edges2bowtie

In [15]:
from sam_bowtie import block_and_fluxes as bnf

## Load data

In [16]:
DATA_FOLDER=HOME+'dati_elezioni/'
TEST_FOLDER=HOME+'tests/'
PVALUE_FOLDER=HOME+'pvalues/'

### Original data

In [17]:
files=os.listdir(DATA_FOLDER)
files.sort()

In [18]:
files

['crisi_dicos.csv',
 'crisi_weighted_edgelist.csv',
 'ita_elections_dicos.csv',
 'ita_elections_weighted_edgelist.csv',
 'quirinale_dicos.csv',
 'quirinale_weighted_edgelist.csv']

In [19]:
l_dataset=len(files)//2

# P-value validation

## FDR function

In [ ]:
def fdr(p_vals_list, alpha=0.01):
    p_vals_array = np.array(p_vals_list)
    sorted_indices = np.argsort(p_vals_array)
    sorted_p_vals = p_vals_array[sorted_indices]
    m = len(p_vals_array)
    threshold = np.arange(1, m + 1) * alpha / m
    below_threshold = sorted_p_vals <= threshold
    if not np.any(below_threshold):
        return 0
    max_index = np.max(np.where(below_threshold))
    return threshold[max_index]

## P-value validation

### Fluxes

In [23]:
p_vals_files=os.listdir(PVALUE_FOLDER)
p_vals_files.sort()

In [78]:
for p_val_file in p_vals_files:
    if p_val_file.endswith('fluxes.pkl'):
        with open(PVALUE_FOLDER + '/' + p_val_file, 'rb') as f:
            _p_vals = pickle.load(f)
        p_vals_list=[]
        for key, value in _p_vals.items():
            p_vals_list.append(value['p_value'])
        threshold = fdr(p_vals_list, alpha=ALPHA)
        p_vals_list = np.array(p_vals_list)
        validated_ratio=np.sum(p_vals_list<=threshold)
        print(f"{p_val_file.replace('_pvalues_fluxes.pkl', '')}: validated_ratio={validated_ratio/len(p_vals_list):.3f}")

        

crisi_dico0: validated_ratio=0.980
crisi_dico1: validated_ratio=0.898
crisi_dico2: validated_ratio=0.980
crisi_dico3: validated_ratio=0.867
crisi_dico4: validated_ratio=0.778
ita_elections_dico0: validated_ratio=0.898
ita_elections_dico1: validated_ratio=0.939
ita_elections_dico2: validated_ratio=0.918
ita_elections_dico3: validated_ratio=1.000
ita_elections_dico4: validated_ratio=0.796
ita_elections_dico5: validated_ratio=0.833
ita_elections_dico6: validated_ratio=0.833
quirinale_dico0: validated_ratio=0.918
quirinale_dico1: validated_ratio=0.810
quirinale_dico2: validated_ratio=0.878
quirinale_dico3: validated_ratio=0.800
quirinale_dico4: validated_ratio=0.918
quirinale_dico5: validated_ratio=0.857
quirinale_dico6: validated_ratio=0.776


In [82]:
for p_val_file in p_vals_files:
    if p_val_file.endswith('blocks.pkl'):
        with open(PVALUE_FOLDER + '/' + p_val_file, 'rb') as f:
            _p_vals = pickle.load(f)
        p_vals_list=[]
        for key, value in _p_vals.items():
            p_vals_list.append(value['p_value'])
        threshold = fdr(p_vals_list, alpha=ALPHA)
        p_vals_list = np.array(p_vals_list)
        validated_ratio=np.sum(p_vals_list<=threshold)
        print(f"{p_val_file.replace('_pvalues_blocks.pkl', '')}: validated_ratio={validated_ratio/len(p_vals_list):.3f}")

        

crisi_dico0: validated_ratio=1.000
crisi_dico1: validated_ratio=1.000
crisi_dico2: validated_ratio=0.857
crisi_dico3: validated_ratio=0.429
crisi_dico4: validated_ratio=0.571
ita_elections_dico0: validated_ratio=0.857
ita_elections_dico1: validated_ratio=0.714
ita_elections_dico2: validated_ratio=0.857
ita_elections_dico3: validated_ratio=1.000
ita_elections_dico4: validated_ratio=0.429
ita_elections_dico5: validated_ratio=0.714
ita_elections_dico6: validated_ratio=0.429
quirinale_dico0: validated_ratio=0.857
quirinale_dico1: validated_ratio=0.571
quirinale_dico2: validated_ratio=0.714
quirinale_dico3: validated_ratio=0.429
quirinale_dico4: validated_ratio=0.857
quirinale_dico5: validated_ratio=0.286
quirinale_dico6: validated_ratio=0.000


# Debug on the first DiCo

In [65]:
i=0

In [66]:
dico_file = files[2*i]
el_file = files[2*i + 1]
dataset_name=dico_file[:-10]
print(f'[{dt.datetime.now():%Y-%m-%d %H:%M:%S}] ***{dataset_name.title()}***')

# Load the DiCo information
dico=np.genfromtxt(DATA_FOLDER+dico_file, delimiter=',',skip_header=1, autostrip=True, dtype=[('user_id', '>U50'), ('dico', '>U2'), ('h_dico', 'U2'), ('i_dico', 'U2')])

# Load the edge list
el=np.genfromtxt(DATA_FOLDER+el_file, delimiter=',', skip_header=1,autostrip=True, dtype=[('source_id', '>U50'), ('target_id', '>U20'),('weight', 'i4')])

# Select correct dicos
dico_dict={}
bad_dicos=[]
for d in dico:
    if d['dico'].isnumeric():
        dico_dict[d['user_id']]=int(d['dico'])
    else:
        if d['dico'] not in bad_dicos:
            bad_dicos.append(d['dico'])

# Nodes
n_nodes=np.concatenate((el['source_id'], el['target_id']))
n_nodes=np.unique(n_nodes)

# Edges

_tmp = defaultdict(list)
# auxiliary defaultdict to group edges by dico class
for edge in el:
    src = edge['source_id'].strip()
    tgt = edge['target_id'].strip()
    d_src = dico_dict.get(src)
    if d_src is not None and d_src == dico_dict.get(tgt):
        _tmp[d_src].append(edge)

el_dico = defaultdict(
    lambda: np.empty(0, dtype=el.dtype),
    {k: np.array(v, dtype=el.dtype) for k, v in _tmp.items()}
)

del _tmp

dicos=list(el_dico.keys())
dicos.sort()
        

[2026-05-12 14:01:15] ***Crisi***


In [67]:
dicos

[0, 1, 2, 3, 4]

In [68]:
d=1

In [69]:
weighted_el=el_dico[dicos[d]]

In [73]:
topo_el = [(s, t) for s, t, w in weighted_el]
bowtie_dict = edges2bowtie(topo_el)

In [74]:
bowtie_dict.values()

dict_values([np.str_('OUT'), np.str_('OUT'), np.str_('INTENDRILS'), np.str_('INTENDRILS'), np.str_('OUT'), np.str_('OUT'), np.str_('OUT'), np.str_('OUT'), np.str_('SCC'), np.str_('OUT'), np.str_('IN'), np.str_('INTENDRILS'), np.str_('OUT'), np.str_('SCC'), np.str_('OUT'), np.str_('OUTTENDRILS'), np.str_('OUT'), np.str_('OUT'), np.str_('INTENDRILS'), np.str_('OUT'), np.str_('SCC'), np.str_('INTENDRILS'), np.str_('IN'), np.str_('INTENDRILS'), np.str_('INTENDRILS'), np.str_('OUT'), np.str_('OUT'), np.str_('SCC'), np.str_('SCC'), np.str_('OUT'), np.str_('OUT'), np.str_('OUT'), np.str_('SCC'), np.str_('OUT'), np.str_('OUT'), np.str_('INTENDRILS'), np.str_('OUT'), np.str_('OUT'), np.str_('SCC'), np.str_('OUT'), np.str_('OUTTENDRILS'), np.str_('OUT'), np.str_('INTENDRILS'), np.str_('SCC'), np.str_('OUT'), np.str_('SCC'), np.str_('OTHERS'), np.str_('OUT'), np.str_('SCC'), np.str_('OUT'), np.str_('OUT'), np.str_('INTENDRILS'), np.str_('SCC'), np.str_('SCC'), np.str_('OUT'), np.str_('SCC'), np.s

In [70]:
cacca=bnf(weighted_el)

In [71]:
cacca[0]

defaultdict(int,
            {np.str_('OUT'): 17154,
             np.str_('INTENDRILS'): 5814,
             np.str_('SCC'): 4332,
             np.str_('IN'): 2480,
             np.str_('OUTTENDRILS'): 1011,
             np.str_('OTHERS'): 977,
             np.str_('TUBES'): 106})

In [72]:
cacca[1]

defaultdict(int,
            {(np.str_('IN'), np.str_('OUT')): np.int32(90337),
             (np.str_('IN'), np.str_('SCC')): np.int32(66737),
             (np.str_('IN'), np.str_('INTENDRILS')): np.int32(9739),
             (np.str_('IN'), np.str_('IN')): np.int32(1031),
             (np.str_('IN'), np.str_('TUBES')): np.int32(149),
             (np.str_('SCC'), np.str_('SCC')): np.int32(159518),
             (np.str_('SCC'), np.str_('OUT')): np.int32(192857),
             (np.str_('OUT'), np.str_('OUT')): np.int32(1868),
             (np.str_('OUTTENDRILS'), np.str_('OUT')): np.int32(1363),
             (np.str_('OUTTENDRILS'), np.str_('OTHERS')): np.int32(752),
             (np.str_('OUTTENDRILS'), np.str_('INTENDRILS')): np.int32(51),
             (np.str_('OUTTENDRILS'), np.str_('OUTTENDRILS')): np.int32(45),
             (np.str_('OTHERS'), np.str_('OTHERS')): np.int32(237),
             (np.str_('OTHERS'), np.str_('INTENDRILS')): np.int32(69),
             (np.str_('INTENDRILS')

In [47]:
qdecm_filename=HOME+f'/tests/{dataset_name}_dico{d}_qdecm.pkl'

In [48]:
qdecm_filename

'/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/bowtie2//tests/crisi_dico1_qdecm.pkl'

In [ ]:
all_nodes=np.concatenate((el_dico[dicos[d]]['source_id'], el_dico[dicos[d]]['target_id']))
all_nodes=np.unique(all_nodes)

In [49]:
with open(qdecm_filename, 'rb') as f:
    qdecm=pickle.load(f)
                

In [63]:
sampled_wel = qdecm.sample()
sampled_wel=[(all_nodes[edge[0]], all_nodes[edge[1]], edge[2]) for edge in sampled_wel]

In [64]:
sampled_wel

[('1000144495499993089', '1040828736', 1),
 ('1000144495499993089', '1041831000', 2),
 ('1000144495499993089', '1042906497129361408', 1),
 ('1000144495499993089', '1063923588', 1),
 ('1000144495499993089', '1085846191307407360', 1),
 ('1000144495499993089', '1087476792007495680', 1),
 ('1000144495499993089', '1087849336241905664', 1),
 ('1000144495499993089', '1089107530247221249', 1),
 ('1000144495499993089', '1094253615420706816', 1),
 ('1000144495499993089', '110528552', 1),
 ('1000144495499993089', '1119689422432292866', 1),
 ('1000144495499993089', '1125440833099112448', 2),
 ('1000144495499993089', '1140517423', 1),
 ('1000144495499993089', '1141368228515631104', 1),
 ('1000144495499993089', '1142125553245442048', 1),
 ('1000144495499993089', '1150751883566493696', 1),
 ('1000144495499993089', '1160196692135821312', 1),
 ('1000144495499993089', '1165252521448935424', 1),
 ('1000144495499993089', '1166765394977275904', 1),
 ('1000144495499993089', '1168701144', 1),
 ('100014449549

In [65]:
sim_blocks, sim_fluxes = bnf(sampled_wel)

In [66]:
sim_blocks

defaultdict(int,
            {'OUT': 17809,
             'SCC': 3866,
             'IN': 2271,
             'OUTTENDRILS': 738,
             'INTENDRILS': 2132,
             'TUBES': 45,
             'OTHERS': 34})

In [67]:
cacca[0]

defaultdict(int,
            {'OUT': 17154,
             'INTENDRILS': 5814,
             'SCC': 4332,
             'IN': 2480,
             'OUTTENDRILS': 1011,
             'OTHERS': 977,
             'TUBES': 106})